# 03b · Sequence architecture and objective

**pragmatiq series, notebook 3 of 4 after the [tour](00_blueprint_v2_tour.ipynb). Slots 2 and 3: how a history
becomes one state, and what pretraining asks that state to predict.**

Blueprint v2 is a *hybrid*. It borrows PRAGMA's idea of compressing every
event, and the profile, into one vector with a small bidirectional encoder.
It then keeps v1's causal decoder, which reads those vectors in time order
and predicts the next event. The state after the last event depends only on
the profile and the events up to the cutoff, by construction.

This notebook loads the pretrained checkpoint beside an untrained copy of the
same network. It then measures what pretraining taught the model, field by
field, and runs a short pretraining so you can watch the loss fall.

About 5 minutes on the lab GPU, on the tour's 4,000-user sample.

> **Smoke mode.** CI runs this notebook with `TFM_NOTEBOOK_SMOKE=1`: a
> generated 120-user book, an untrained model, and CPU only. Every cell runs,
> but the numbers and plots are meaningless. The first code cell prints which
> mode you are in.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from tfm.runtime.lab import fetch_lab_assets, load_lab_model, prepare_lab_data
from tfm.training.reproducibility import seed_everything

GREEN, DARK, GREY = "#76B900", "#1A1A1A", "#8C8C8C"
PALETTE = [GREEN, "#0074DF", "#F2A900", "#5D1682", "#E52020", DARK, GREY, "#00A3A3"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

assets = fetch_lab_assets("03")
config = assets.config
seed_everything(config.seed)
print(assets.describe())

In [ ]:
data = prepare_lab_data(assets)
model = load_lab_model(assets, data)
random_model = load_lab_model(assets, data, pretrained=False)
g = config.model.composition.granularity
print(f"hidden {g.decoder.hidden_size}, {g.decoder.num_heads} heads; depths: profile {g.profile_depth}, "
      f"event {g.event_depth}, decoder {g.decoder.depth}")

## The network

```text
profile tokens (as of cutoff) -> profile encoder (bidirectional) -> z_profile
event j key/value tokens      -> event encoder (bidirectional)   -> z_event[j]
[z_profile, z_event[0..N-1]] + calendar, TimeRoPE -> causal decoder -> h[0..N]
```

Each event is encoded on its own, so the expensive attention over tokens
stays inside an event of at most a few tokens. The decoder attends over one
vector per event, which is why it can read 128 events cheaply.

In [ ]:
from matplotlib.patches import FancyBboxPatch

W, H = 0.2, 0.2
boxes = {
    "profile": (0.0, 0.74, "Profile tokens\n(as of the cutoff)", GREY),
    "events": (0.0, 0.32, "Event j's key/value tokens\n(each event on its own)", GREY),
    "p_enc": (0.26, 0.74, f"Profile encoder\nbidirectional\n{g.profile_depth} layers", "#0074DF"),
    "e_enc": (0.26, 0.32, f"Event encoder\nbidirectional inside the event\n{g.event_depth} layers", "#0074DF"),
    "dec": (0.52, 0.53, f"Causal decoder, {g.decoder.depth} layers\nslots: profile, event 0 … N-1\n"
                        "+ calendar, TimeRoPE(Δt)", GREEN),
    "next": (0.8, 0.74, "Next-event heads\nslot j predicts every\nfield of event j", "#F2A900"),
    "emb": (0.8, 0.32, "Readout: the slot after\nthe last event is the\nuser's embedding", "#F2A900"),
    "mlm": (0.26, 0.02, "Masked-token loss\ninside each event (MLM)", "#F2A900"),
}
links = [("profile", "p_enc", ""), ("events", "e_enc", ""), ("p_enc", "dec", "z_profile"),
         ("e_enc", "dec", "z_event[j]"), ("dec", "next", "h[j]"), ("dec", "emb", "h[N]"),
         ("e_enc", "mlm", "")]

fig, ax = plt.subplots(figsize=(13, 5))
ax.set_axis_off()
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(0, 1)
for x, y, text, colour in boxes.values():
    ax.add_patch(FancyBboxPatch((x, y), W, H, boxstyle="round,pad=0.01", fc=colour, alpha=0.15,
                                ec=colour, lw=1.5))
    ax.text(x + W / 2, y + H / 2, text, ha="center", va="center", fontsize=9)
for a, b, label in links:
    (xa, ya, *_), (xb, yb, *_) = boxes[a], boxes[b]
    start, end = ((xa + W, ya + H / 2), (xb, yb + H / 2)) if xb > xa else ((xa + W / 2, ya), (xb + W / 2, yb + H))
    ax.annotate("", end, start, arrowprops={"arrowstyle": "-|>", "color": DARK, "lw": 1.2})
    if label:
        ax.text((start[0] + end[0]) / 2, (start[1] + end[1]) / 2 + 0.03, label, ha="center",
                fontsize=8, style="italic")
ax.set_title("Blueprint v2 hybrid: blue encodes, green reads the history, amber is trained or read out")
plt.show()

### Where the parameters are

The token embedding table is the largest single module: 12,870 tokens × 256
dimensions, most of them hash buckets. The three transformer stacks share
the rest.

In [ ]:
params = {name: sum(p.numel() for p in module.parameters()) for name, module in model.named_children()}
params = pd.Series({k: v for k, v in params.items() if v}).sort_values()
fig, ax = plt.subplots(figsize=(10, 0.3 * len(params) + 1.2))
ax.barh(params.index, params.to_numpy(), color=GREEN)
for i, n in enumerate(params.to_numpy()):
    ax.annotate(f" {n / 1e6:.2f}M", (n, i), va="center", fontsize=8)
ax.set_xlabel("parameters")
ax.set_title(f"{params.sum() / 1e6:.1f}M parameters by module")
plt.tight_layout()
plt.show()

### Who attends to whom

Two attention patterns, drawn from one real user's first events. Inside the
event encoder, a token attends to every token of its own event and to
nothing else (left). In the decoder, slot 0 is the profile and slot `j + 1`
is event `j`, and each slot attends only to itself and earlier slots
(right). That causal mask is what makes the embedding at the cutoff safe to
use for forecasting.

In [ ]:
record = data.tokenizer.encode(data.test_sequences[0])
offsets = np.asarray(record.event_offsets[:7]) - record.event_offsets[0]
within = np.zeros((offsets[-1], offsets[-1]))
for a, b in zip(offsets, offsets[1:]):
    within[a:b, a:b] = 1
slots = 13
causal = np.tril(np.ones((slots, slots)))

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.6))
left.imshow(within, cmap="Greens", vmin=0, vmax=1.4)
left.set_title("Event encoder: tokens of 6 events")
left.set_xlabel("attends to token")
left.set_ylabel("token")
right.imshow(causal, cmap="Greens", vmin=0, vmax=1.4)
right.set_xticks(range(slots), ["P"] + [str(j) for j in range(slots - 1)])
right.set_yticks(range(slots), ["P"] + [str(j) for j in range(slots - 1)])
right.set_title("Decoder: profile (P) and events 0-11")
right.set_xlabel("attends to slot")
for axis in (left, right):
    axis.grid(False)
plt.tight_layout()
plt.show()

## The objective

`next_event_fields`: the decoder state that has seen the profile and events
`0…j-1` predicts every field value of event `j`. The head is told which keys
event `j` has, so it predicts *values*, not which fields will appear. The
auxiliary `token_event_key_mlm` (PRAGMA's objective) masks tokens, whole
events, and whole keys inside each event, and asks the event encoder to
restore them. The losses are summed.

Scored on 256 held-out validation users, pretraining has cut every loss by
an order of magnitude against the same network untrained:

In [ ]:
from tfm.adaptation.batching import minibatches, select_rows

held_out = select_rows(data.batches[1], list(range(min(256, data.batches[1].batch_size))))
losses = {}
for name, m in {"pretrained": model, "random init": random_model}.items():
    torch.manual_seed(config.seed)
    parts = []
    with torch.no_grad():
        for chunk in minibatches(held_out, 16):
            parts.append({k: float(v) for k, v in m(chunk.to(assets.device)).losses.items()})
    losses[name] = pd.DataFrame(parts).mean()
losses = pd.DataFrame(losses)

fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(len(losses))
ax.bar(x - 0.2, losses["random init"], 0.4, color=GREY, label="random init")
ax.bar(x + 0.2, losses["pretrained"], 0.4, color=GREEN, label="pretrained")
ax.set_xticks(x, losses.index)
ax.set_yscale("log")
ax.set_ylabel("loss (log scale)")
ax.set_title("Held-out loss per objective")
ax.legend()
plt.tight_layout()
plt.show()
losses.round(3)

### What the model learned, field by field

The total hides which fields are predictable. `next_event_ce_by_key` gives
the cross-entropy of the true next value for each key, in nats (lower is
better). It is set against two floors, measured on 2,000 users of the same book
([recorded](../../benchmarks/results/blueprint_v2_hybrid_gb10_tuning.md); the checkpoint ships them as `next_event_floors.json`):

- **unigram**: predict the key's train-side value frequencies, ignoring the user;
- **copy**: repeat the user's previous value of that key.

The pretrained model beats both floors on `amount`, `mcc`, and
`app_version`, and comes close to the copy floor on `device_id` (people keep
their phones). `currency` and `country` are nearly free. `merchant`, the
field that says most about a card payment, is only just below its unigram
floor on this checkpoint (about 6.4 against 6.58 nats) after 3,000 steps:
learnable, slowly. Two keys are unresolved. `counterparty` is worse than
either floor. Every `campaign_id` after the cutoff is a value unseen in
training, so its loss measures that shift, not the model.

In [ ]:
from tfm.evaluation.surprise import next_event_ce_by_key

from tfm.runtime.common import repo_root

tuning = json.loads((repo_root() / "benchmarks/results/blueprint_v2_hybrid_gb10_tuning.json").read_text())
floors = pd.DataFrame(tuning["diagnostics"]["diagnostics_tokens_hkey4096"]["keys"]).T
ce = pd.DataFrame({name: next_event_ce_by_key(m, held_out, data.tokenizer, device=assets.device)
                   for name, m in {"pretrained": model, "random init": random_model}.items()})
ce = ce.join(floors[["unigram_ce", "copy_ce"]].astype(float)).sort_values("pretrained")

fig, ax = plt.subplots(figsize=(11, 0.32 * len(ce) + 1.4))
y = np.arange(len(ce))
ax.barh(y, ce["pretrained"], color=GREEN, label="pretrained (this sample)")
ax.scatter(ce["unigram_ce"], y, color=DARK, marker="|", s=120, label="unigram floor")
ax.scatter(ce["copy_ce"], y, color="#E52020", marker="|", s=120, label="copy floor")
ax.set_yticks(y, ce.index)
ax.set_xlabel("next-event cross-entropy (nats)")
ax.set_title("Per-key next-event loss; the untrained model scores "
             f"{ce['random init'].median():.0f} nats at the median")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

## Pairings the kit refuses

Most slot choices combine freely. Four rules do not, because breaking them
silently produces a model that cheats or cannot learn:

1. A causal objective needs a causal history model.
2. Masked-token objectives need bidirectional attention inside their unit.
3. The readout must see every event up to the cutoff.
4. A structure that encodes events one at a time needs event boundaries.

For example, next-event prediction on a bidirectional history encoder would
let every slot see the event it predicts:

In [ ]:
from tfm import EagerAttention, LastTokenPool
from tfm.contracts.errors import IncompatibleCompositionError
from tfm.experimental import NextEventFields, ProfileHistory, TimeRoPE, validate_composition

try:
    validate_composition(
        ProfileHistory(vocab_size=64), TimeRoPE(), EagerAttention(),
        NextEventFields(), LastTokenPool(), None,
    )
except IncompatibleCompositionError as error:
    print("Refused:", error)

## Pretrain for a few minutes

The checkpoint trained for 3,000 steps of 128 windows, about 42 minutes on a
GB10, over every 128-event train-side window of all 100,000 users, read
from a token cache. Below, an untrained copy trains on this notebook's
4,000-user sample with the same optimizer: Muon on the hidden matrices,
AdamW on the rest, cosine schedule. `loss_log` records every step's loss.

In [ ]:
from tfm.runtime.lab import lab_run
from tfm.runtime.pipeline import training_batches
from tfm.training.trainer import train_model

scratch = load_lab_model(assets, data, pretrained=False).train()
batches, _ = training_batches(lab_run(assets, data, scratch))
curve = []
started = time.perf_counter()
stats = train_model(
    scratch, batches, max_steps=assets.train_steps, learning_rate=config.training.learning_rate,
    seed=config.seed, device=assets.device, optimizer=config.training.optimizer, shuffle=True,
    precision=config.training.precision, loss_log=curve,
)
print(f"{len(curve)} steps in {time.perf_counter() - started:.0f} s; "
      f"loss {stats['first_loss']:.2f} -> {stats['final_loss']:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(np.arange(1, len(curve) + 1), curve, color=GREY, alpha=0.5, label="per step")
window = max(1, len(curve) // 15)
ax.plot(np.arange(window, len(curve) + 1), np.convolve(curve, np.ones(window) / window, "valid"),
        color=GREEN, lw=2, label=f"{window}-step mean")
if assets.checkpoint_dir is not None:
    stats = json.loads((assets.checkpoint_dir / "train_stats.json").read_text())
    ax.axhline(stats["loss_tail_mean"], color=DARK, ls="--",
               label=f"checkpoint at step {int(stats['steps']):,}: {stats['loss_tail_mean']:.2f}")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
ax.yaxis.set_minor_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("step")
ax.set_ylabel("training loss (log scale)")
ax.set_title("Pretraining from scratch on the lab sample")
ax.legend()
plt.tight_layout()
plt.show()

## How the checkpoint was configured

The tokens, model size, and optimizer above came from an earlier
[staged search](../../benchmarks/results/blueprint_v2_hybrid_gb10_tuning.md) on one GB10. A final sweep then settled two numbers: how long to
pretrain (1,500 or 3,000 steps) and the LoRA rank used to fine-tune each
task (8 or 16, notebook 04). It chose by downstream results, how much the
fine-tuned model lifted the tasks over a raw-feature baseline, not by
pretraining loss. The lab's checkpoint is the 3,000-step run, fine-tuned at
rank 16 ([record](../../benchmarks/results/blueprint_v2_final_sweep.md)). Its config, `benchmarks/ablations/blueprint_v2_final_d.yaml`,
differs from the recipe's `config.yaml` in four lines: `training.max_steps` (3,000), `training.optimizer.warmup_steps`
(150), and `fraud_task.finetune.rank` and `alpha` (16).

## Other objectives in the kit

`objective.auxiliary` can also add `past_reconstruction` (predict the fields
of the previous few events) and `subsequence_contrastive` (a CoLES-style loss
that pulls two random crops of the same history together). Both are
experimental; see `tfm.config`.

### Exercises

1. Train for longer: run `import dataclasses; assets = dataclasses.replace(assets, train_steps=600)` before the pretraining cell and compare your curve with the checkpoint's dashed line. Where does the gap close fastest?
2. In a copy of the checkpoint's `config.yaml` (in the `TFM_V2_CHECKPOINT` directory), remove `token_event_key_mlm` from `objective.auxiliary` and pretrain again. Which held-out loss changes, and does the per-key next-event loss move?
3. Try the other refused pairings: call `validate_composition` with a masked objective on a causal decoder, or a readout that can't see the last event, and read the reason the SDK gives.

## Where this lives in the kit

This notebook explains; the recipe runs. Every cell calls `src/tfm`, so the
same code runs in [`recipes/blueprint_v2`](../../recipes/blueprint_v2), and no
reusable logic lives here.

The model is `tfm.layers.profile_event_decoder`, composed by
`tfm.experimental.compose_blueprint_v2`. The objective is
`tfm.objectives.next_event_fields`. A few minutes on 4,000 users is not the
checkpoint's run; the header of the checkpoint's `config.yaml` has the commands for that.

Next: [04 · Readout and adaptation](04_readout_and_adaptation.ipynb).